In [2]:
from datasets.dataset_dict import DatasetDict
from datasets import Dataset, concatenate_datasets
import evaluate

import pandas as pd
import numpy as np
from datetime import datetime
from sklearn.model_selection import train_test_split 
from transformers import pipeline, AutoTokenizer, AutoModel, AutoConfig, PretrainedConfig, PreTrainedModel
import os
import torch
import torch.nn as nn
from transformers.modeling_outputs import SequenceClassifierOutput
from sklearn.metrics import f1_score, classification_report
from transformers.models.auto import auto_factory

NOTE this code is messy and is purely for checking the models made by John, if you do not follow what is happening here, that is okay, because this is not critical. For more streamlined model testing, consider running the golden_dataset app, or, more aptly, check out HRAF_NLP/HRAF_MultiLabel_SubClasses_Kfolds jupyter notebooks using Model 5 as this is the best model

In [161]:
# from huggingface_hub import notebook_login
# # If below code does not work, copy and paste this code in the terminal: huggingface-cli login 
# then paste this read token (you will have to construct it)


# notebook_login()

In [3]:
# load a list of passages and predict them (will take about .25 seconds per passage for me so beware the wait)
def predictor(data, labels, tokenizer_kwargs, classifier):
    dataOutput = []
    for text in data:
        # get actual labels
        actual_labels = [text[label] for label in labels]
        prediction = classifier(text['passage'], **tokenizer_kwargs)

        # get predicted labels
        scores = {item['label']:item['score'] for item in prediction[0]} #turn prediction into a dictionary
        pred_labels = [1 if scores[label] >= 0.5 else 0 for label in labels]

        
        output_dict = dict()
        output_dict["pred_labels"] = pred_labels
        output_dict["actual_labels"] = actual_labels
        output_dict["passage"] = text['passage']
        output_dict["ID"] = text['ID']


        # score[0][("actual_label", 'passage')] = text['passage'], text['label']
        dataOutput.append(output_dict)
    return dataOutput

# Get F1 scores
def score(dataOutput, labels):
    from sklearn.metrics import f1_score, accuracy_score

    df_score = pd.DataFrame(index=['NLP'], columns= [label+"_F1" for label in labels] + ["Micro_F1", "Macro_F1"])
    actual_labels = [x['actual_labels'] for x in dataOutput]
    pred_labels = [x['pred_labels'] for x in dataOutput]
    for index, label in enumerate(labels):
        f1 = round(f1_score(y_true=np.array(actual_labels)[:,index], y_pred=np.array(pred_labels)[:,index]),3)
        df_score.at['NLP', label+"_F1"] = f1
        # print(f"{label}: {(6 - len(label)) *' '}{f1}")

    # print("\n")

    f1_micro = round(f1_score(y_true=actual_labels, y_pred=pred_labels, average='micro'),3)
    f1_macro = round(f1_score(y_true=actual_labels, y_pred=pred_labels, average='macro'),3)
    f1_weighted = round(f1_score(y_true=actual_labels, y_pred=pred_labels, average='weighted'),3)
    df_score.at['NLP', "Micro_F1"] = f1_micro
    df_score.at['NLP', "Macro_F1"] = f1_macro
    df_score.at['NLP', "Weighted_F1"] = f1_weighted
    return df_score
def cor_score(dataOutput, labels):
    from sklearn.metrics import matthews_corrcoef
    df_score = pd.DataFrame(index=['NLP'], columns= [label+"_Cor" for label in labels])
    actual_labels = [x['actual_labels'] for x in dataOutput]
    pred_labels = [x['pred_labels'] for x in dataOutput]
    for index, label in enumerate(labels):
        corrcoef = round(matthews_corrcoef(y_true=np.array(actual_labels)[:,index], y_pred=np.array(pred_labels)[:,index]),3)
        df_score.at['NLP', label+"_Cor"] = corrcoef    
    return df_score
    # print(f'F1 score (micro) {f1_micro}\nF1 score (macro) {f1_macro}')

## Inference

### Load datset 

In [5]:
import json
loc = "../HRAF_NLP/HRAF_MultiLabel_SubClasses_Kfolds/"
# loc = "../HRAF_MultiLabel_ThreeLargeClasses/" #load old threemain class (comment this out unless you specifically are using it)

# dataset = load_dataset('csv', data_files={'train': 'train.txt', 'validation': 'val.txt', 'test': 'test.txt'}, sep=";", 
#                               names=["text", "label"])


# f = open(loc+"Datasets/John Test Datasets/test_dataset.json") #John's dataset
f = open(loc+"Datasets/test_dataset.json")
# f = open("../HRAF_MultiLabel_ThreeLargeClasses/Datasets/test_dataset.json") #load old threemain class (comment this out unless you specifically are using it)
data = json.load(f)
f.close()
Hraf = Dataset.from_dict(data)
Hraf

Dataset({
    features: ['ID', 'passage', 'EVENT_Illness', 'EVENT_Accident', 'EVENT_Other', 'CAUSE_Material_Physical', 'CAUSE_Spirits_Gods', 'CAUSE_Witchcraft_Sorcery', 'CAUSE_Rule_Violation_Taboo', 'ACTION_Physical_Material', 'ACTION_Technical_Specialist', 'ACTION_Divination', 'ACTION_Shaman_Medium_Healer', 'ACTION_Priest_High_Religion'],
    num_rows: 2074
})

### Get John Data

In [6]:
# Get John Dataset
path = "data/objects/tiered/tiered_scored_embedded_cleaned_raw__Altogether_Dataset_RACoded_Combined_20251014_091039/"

# CHOOSE 1

# # Option 1: Tier 1+2 dataset
# df_tier1 = pd.read_excel(path+"tier1.xlsx")
# df_tier2 = pd.read_excel(path+"tier2.xlsx")
# df_John = pd.concat([df_tier1,df_tier2])
# print(len(df_John))
# straightJohn = False #Here to make sure certain lines do not get run
# df_John.head(2)


#Option 2: Straight John Inference.
# df_John = pd.read_excel(path+"inference.xlsx")
# print(len(df_John))
# straightJohn = True #Here to make sure certain lines do not get run

# # Option 3: None of John's tier data within the original Eric test dataset
# # NOTE, run this chunk then skip to "Define Kwargs and Labels"
df_tier1 = pd.read_excel(path+"tier1.xlsx")
df_tier2 = pd.read_excel(path+"tier2.xlsx")
df_John = pd.concat([df_tier1,df_tier2])
df_John.head(2)
# Get original DF to match with John's
passages_John = df_John["Passage Number"]
# get indexes of matching values
index_list = []
for i, pass_n in enumerate(Hraf['ID']):
    if pass_n not in passages_John:
        index_list.append(i)
Hraf = Dataset.from_dict(Hraf[index_list]) # index the passage numbers that do not intersect with Johns, as wella s turn the dataset back to a huggingface dataset
len(Hraf)


1970

In [8]:

df_path = "../Coding and Dataset/Data/"
# df_path = "../../../eHRAF_Scraper-Analysis-and-Prep/Data/"
dataFolder = r"(subjects-(contracts_OR_disabilities_OR_disasters_OR_friendships_OR_gift_giving_OR_infant_feeding_OR_lineages_OR_etc/"
# dataFolder = r'subjects-(sickness)_FILTERS-culture_level_samples(PSF)'


# Get model and centralized path if relevent
model_name = "HRAF_MultiLabel_SubClasses_Kfolds"
path = f"" #Path to centralized file locations (leave blank if centralized location is here)



# load df (only load one of these commented out lines)
# df = pd.read_excel(f"{df_path}{dataFolder}/_Altogether_Dataset_RACoded.xlsx", header=[0,1], index_col=0) # Fall 2023 sickness + non-sickness
df = pd.read_excel(f"{df_path}{dataFolder}/_Altogether_Dataset_RACoded_Combined.xlsx", header=[0,1], index_col=0) # Spring 2023 - Spring 2024  sickness + nonsickness dataset
print(len(df))
print(df[("CODER","Run_Number")].value_counts())
df.head(2)


11005
(CODER, Run_Number)
3    9028
1    1926
2      51
Name: count, dtype: int64


CULTURE                               \
  Passage Number Region   SubRegion   Culture   
0           1392   Asia  South Asia  Andamans   
1           1393   Asia  South Asia  Andamans   

                                                                     \
                                            DocTitle        Section   
0  Hygiene and medical practices among the Onge (...  1. Habitation   
1  Hygiene and medical practices among the Onge (...        3. Food   

                               \
            Author Page  Year   
0  Cipriani, Lidio  484  1961   
1  Cipriani, Lidio  487  1961   

                                                      ... ACTION  \
                                                 OCM  ...  Other   
0  ['171', '301', '727', '751', '765', '775', '777']  ...      1   
1  ['136', '231', '271', '312', '415', '516', '751']  ...      0   

                                                      \
                                         Description   
0  Several customs are believed to connect with t...   
1                             No action is mentioned   

                                                      \
                                         Local_terms   
0   ibidanghe: made from decorated human jawbone ...   
1                                                  0   

                                               OTHER      CODER           \
                                      Other_Comments Run_Number Finished   
0  General note of this spreadsheet - many of the...          1     True   
1                                                NaN          1     True   

                   OTHER   CODER  \
  Coder Other_Comments.1 Dataset   
0    YM              NaN       1   
1    YM              NaN       1   

                                                      
                                                Info  
0  Dataset 1: ['750', '751', '752', '753']   Coun...  
1  Dataset 2: ['784', '731', '732', '777', '791',...  

[2 rows x 43 columns]

In [6]:
### Check for duplicates in run 2 which do not appear in BOTH run 1 and 3 (and thus are eroneously deleted, these are the 20 passages spoken about later)
# df_run1 = df.loc[df[("CODER","Run_Number")]==1]
# df_run2 = df.loc[df[("CODER","Run_Number")]==2]
# df_run3 = df.loc[df[("CODER","Run_Number")]==3]

# all3= df_run2[df_run2[("CULTURE","Passage")].isin(df_run1[("CULTURE","Passage")])][("CULTURE","Passage")].isin(df_run3[("CULTURE","Passage")])
# not3_index = all3[~all3].index
# df_not3 = df.iloc[list(not3_index)]
# df_not3[("CODER","Run_Number")].value_counts()

# # counter = 0
# # for i, value in enumerate(pass_try):
# #     passages = df.loc[df[("CULTURE","Passage")]==value]
# #     run_length =  len(passages)
    
# #     if run_length <3:
# #         counter += 1
# #         print("fail: ", all3_index[i])
# # print(counter)
# # df.loc[df[("CULTURE","Passage")]==pass_try]


In [23]:
# useRuns = [1,3] #Only include these runs (NOTE see note below as well, but THIS IS COMMENTED OUT IN ORDER TO NOT RUIN THE SUBLABEL DATASET BUT EVENTUALLY YOU SHOULD USE THIS CODE)
# df = df.loc[df[("CODER","Run_Number")].isin(useRuns)]

# NOTE: the resulting number of duplicates removed (at time of writing) is 617 yet 637 were removed (not counting the 1 extra special duplicate removed in values_to_remove).
# The extra 20 removed are from run 1 which had duplicates in run 2. These should have been kept and the msitake was due to the way ALL dulicates were marked for removal 
# but only run 3 duplicates were kept. The commented out code above actually would fix this issue, but since we don't want to change the dataset now that everything is trained, we will cannot remove

print("Starting Passages :",len(df))
print("Duplicate Passages:",sum((df.duplicated(("CULTURE","Passage")))))
# mask_NotDuplicate = ~(df.duplicated(("CULTURE","Passage"), keep=False))
mask_NotDuplicate = (df.duplicated(("CULTURE","Passage"), keep="first")) 
mask_Dataset2 = df[("CODER","Run_Number")]==3

# df = df[(mask_NotDuplicate) |  (mask_Dataset2)]
df = df[(~mask_NotDuplicate)]




# Remove certain passages which should not be in training or inference (these are duplicates that had to be manually found by a human)
# NOTE this normally would be ran, but John did not remove these passages and one is part of his chosen 5000 dataset unfortunately... SO, we will not remove
# values_to_remove = [3252, 33681, 6758, 10104]
# df = df[~df[('CULTURE','Passage Number')].isin(values_to_remove)]

print("Passages Remaining :",len(df))
print("Duplicate Passages:",sum((df.duplicated(("CULTURE","Passage")))))
df[("CODER","Run_Number")].value_counts()

Starting Passages : 11005
Duplicate Passages: 617
Passages Remaining : 10388
Duplicate Passages: 0


(CODER, Run_Number)
3    8462
1    1896
2      30
Name: count, dtype: int64

In [24]:
# Get original DF to match with John's
passages_John = df_John["Passage Number"]


df_Eric = df[df[("CULTURE","Passage Number")].isin(passages_John)]
print("Passages Remaining :",len(df_Eric))

Passages Remaining : 2086


In [25]:
df_John_indexed = df_John.set_index("Passage Number", drop=True)
df_Eric_indexed = df_Eric.set_index(("CULTURE","Passage Number"), drop=True)
df_Eric_indexed = df_Eric_indexed.loc[list(df_John_indexed.index)].reset_index()
df_Eric_indexed.head(3)

CULTURE                               \
  Passage Number Region   SubRegion   Culture   
0           1393   Asia  South Asia  Andamans   
1           1412   Asia  South Asia  Andamans   
2           1286   Asia  South Asia  Andamans   

                                                      \
                                            DocTitle   
0  Hygiene and medical practices among the Onge (...   
1  Hygiene and medical practices among the Onge (...   
2  The Andaman islanders: a study in social anthr...   

                                              \
                                     Section   
0                                    3. Food   
1   III. Illnesses and Onge Curative Methods   
2  CHAPTER III RELIGIOUS AND MAGICAL BELIEFS   

                                                         \
                                      Author Page  Year   
0                            Cipriani, Lidio  487  1961   
1                            Cipriani, Lidio  499  1961   
2  Radcliffe-Brown, A. R. (Alfred Reginald),  181  1922   

                                                      ... ACTION  \
                                                 OCM  ...  Other   
0  ['136', '231', '271', '312', '415', '516', '751']  ...      0   
1                                     ['751', '754']  ...      0   
2                                ['0', '751', '824']  ...      1   

                                                                  \
                                         Description Local_terms   
0                             No action is mentioned           0   
1                             No action is mentioned           0   
2  Carrying anadendron paniculatum protects one f...           0   

           OTHER      CODER                           OTHER   CODER  \
  Other_Comments Run_Number Finished Coder Other_Comments.1 Dataset   
0            NaN          1     True    YM              NaN       1   
1            NaN          1     True    YM              NaN       1   
2            NaN          1     True    YM              NaN       1   

                                                      
                                                Info  
0  Dataset 2: ['784', '731', '732', '777', '791',...  
1                                                NaN  
2                                                NaN  

[3 rows x 43 columns]

In [121]:
#Construct col list
cols = list(df.columns)
id_index = cols.index(('CULTURE', "Passage Number"))
passage_index = cols.index(('CULTURE', "Passage"))
event_index =  cols.index(('EVENT', "No_Info"))
cause_index = cols.index(('CAUSE', "No_Info"))
action_index = cols.index(('ACTION', "No_Info"))
# get a list of all the multi-indexed column names we want to evaluate
# col_list = [cols[id_index]] + [cols[passage_index]] + cols[event_index:event_index+4] + cols[cause_index:cause_index+7] + cols[action_index:action_index+7] #to include all columns including No_info
col_list = [cols[id_index]] + [cols[passage_index]] + cols[event_index+1:event_index+4] + cols[cause_index+1:cause_index+7] + cols[action_index+1:action_index+7] # to include al columns BUT No_info


## Remove the following columns from the dataset. Based on the results of previous models ran and the bias of the categories
remv_cols = [("CAUSE","Just_Happens"),("CAUSE","Other"),("ACTION","Other")]
for remv in remv_cols:
    col_list.remove(remv)



# get column names to ascribe to the new data frame
colNames = ["ID","passage"]
for category, sub_cat in col_list:
    # skip passage and id which have already been added
    # print(category, sub_cat)
    if category == "CULTURE":
        continue
    if sub_cat == "No_Info":
        colNames += [category]#this to include main classes, we will hold off on that
        pass
    else:
        colNames += [f'{category}_{sub_cat}']

print("Columns excluded:\n", set(cols)-set(col_list),"\n")
# for col in col_list
print("Columns included:")
for col in colNames:
    print(col)
# colNames


Columns excluded:
 {('CULTURE', 'Page'), ('ACTION', 'Local_terms'), ('CODER', 'Finished'), ('CODER', 'Coder'), ('ACTION', 'Description'), ('CULTURE', 'Year'), ('CULTURE', 'Author'), ('CAUSE', 'Just_Happens'), ('CAUSE', 'Other'), ('CODER', 'Info'), ('OTHER', 'Other_Comments'), ('EVENT', 'Description'), ('ACTION', 'No_Info'), ('CULTURE', 'SubRegion'), ('CULTURE', 'Section'), ('CAUSE', 'Description'), ('CULTURE', 'Culture'), ('CULTURE', 'OCM'), ('EVENT', 'No_Info'), ('CODER', 'Dataset'), ('EVENT', 'Local_Terms'), ('CULTURE', 'OWC'), ('CAUSE', 'No_Info'), ('CAUSE', 'Local_Terms'), ('OTHER', 'Other_Comments.1'), ('CULTURE', 'Region'), ('CULTURE', 'DocTitle'), ('CODER', 'Run_Number'), ('ACTION', 'Other')} 

Columns included:
ID
passage
EVENT_Illness
EVENT_Accident
EVENT_Other
CAUSE_Material_Physical
CAUSE_Spirits_Gods
CAUSE_Witchcraft_Sorcery
CAUSE_Rule_Violation_Taboo
ACTION_Physical_Material
ACTION_Technical_Specialist
ACTION_Divination
ACTION_Shaman_Medium_Healer
ACTION_Priest_High_Religi

In [ ]:
assert straightJohn == False, "this block should not be ran as the dataset loaded is already the inference"
# subdivide into just passage and outcome
df_small = pd.DataFrame()
df_small[colNames] = df_Eric_indexed[col_list]
# Flip the lable of "no_info"
# df_small[["EVENT","CAUSE","ACTION"]]  = df_small[["EVENT","CAUSE","ACTION"]].replace({0:1, 1:0})


# create train and validation/test sets (Also do John's as I want to double check we are splitting the same way)
train_val_E, test_E = train_test_split(df_small, test_size=0.2, random_state=42)
train_val_J, test_J = train_test_split(df_John, test_size=0.2, random_state=42)


# Create an NLP friendly dataset
Hraf = Dataset.from_dict(test_E.to_dict(orient= 'list'))
Hraf

Dataset({
    features: ['ID', 'passage', 'EVENT_Illness', 'EVENT_Accident', 'EVENT_Other', 'CAUSE_Material_Physical', 'CAUSE_Spirits_Gods', 'CAUSE_Witchcraft_Sorcery', 'CAUSE_Rule_Violation_Taboo', 'ACTION_Physical_Material', 'ACTION_Technical_Specialist', 'ACTION_Divination', 'ACTION_Shaman_Medium_Healer', 'ACTION_Priest_High_Religion'],
    num_rows: 1000
})

In [122]:
assert straightJohn == True, "this block should not be ran as the dataset loaded should be split for test/val"
df_small = pd.DataFrame()
df_small[colNames] = df_Eric_indexed[col_list]


# Create an NLP friendly dataset
Hraf = Dataset.from_dict(df_small.to_dict(orient= 'list'))
Hraf

Dataset({
    features: ['ID', 'passage', 'EVENT_Illness', 'EVENT_Accident', 'EVENT_Other', 'CAUSE_Material_Physical', 'CAUSE_Spirits_Gods', 'CAUSE_Witchcraft_Sorcery', 'CAUSE_Rule_Violation_Taboo', 'ACTION_Physical_Material', 'ACTION_Technical_Specialist', 'ACTION_Divination', 'ACTION_Shaman_Medium_Healer', 'ACTION_Priest_High_Religion'],
    num_rows: 2086
})

In [ ]:
# Check to make sure the rows align with the John's (Should give 1000)
assert 1000 == len(test_E[test_E[("ID")].isin(test_J["Passage Number"])]), "Not same test split"
# assert 1000 == len(test_E[test_E[("passage")].isin(test_J["Passage"])]), "Not same test split"

### Define Kwargs and Labels

In [13]:
#Remap names to fit John's naming convention

# The mapping of old names → new names
rename_map = {
    "EVENT_Illness": "Illness",
    "EVENT_Accident": "Accident",
    "EVENT_Other": "Other",
    "CAUSE_Material_Physical": "Material_Physical",
    "CAUSE_Spirits_Gods": "Spirits_Gods",
    "CAUSE_Witchcraft_Sorcery": "Witchcraft_Sorcery",
    "CAUSE_Rule_Violation_Taboo": "Rule_Violation_Taboo",
    "ACTION_Physical_Material": "Physical_Material",
    "ACTION_Technical_Specialist": "Technical_Specialist",
    "ACTION_Divination": "Divination",
    "ACTION_Shaman_Medium_Healer": "Shaman_Medium_Healer",
    "ACTION_Priest_High_Religion": "Priest_High_Religion",
}

# Apply the renaming
for old, new in rename_map.items():
    Hraf = Hraf.rename_column(old, new)

In [14]:
# Define tokenizer kwargs

# get label names
label_columns = [label for label in Hraf.features.keys() if label not in ['ID', 'passage']]
label_columns




['Illness',
 'Accident',
 'Other',
 'Material_Physical',
 'Spirits_Gods',
 'Witchcraft_Sorcery',
 'Rule_Violation_Taboo',
 'Physical_Material',
 'Technical_Specialist',
 'Divination',
 'Shaman_Medium_Healer',
 'Priest_High_Religion']

## John Inference


initialize model parameters

In [11]:
# ============================================================================
# CELL 4: MODEL DEFINITION WITH MAIN LABEL CONTROL
# ============================================================================



class ConfigurableHierarchicalConfig(PretrainedConfig):
    """Configuration for configurable hierarchical model"""
    model_type = "configurable_hierarchical"

    def __init__(
        self,
        base_model="roberta-base",
        use_hierarchy=True,
        gated_hierarchy=True,
        gate_threshold=0.5,
        hidden_size=768,
        hierarchical_hidden_size=256,
        num_hidden_layers=2,
        dropout=0.2,
        attention_dropout=0.1,
        use_weighted_loss=False,
        use_focal_loss=True,
        focal_gamma=2.5,
        teacher_forcing_ratio=0.7,
        predict_main_labels=True,  # NEW parameter
        num_main_labels=3,
        num_event_labels=2,
        num_cause_labels=4,
        num_action_labels=3,
        total_labels=12,
        label_indices=None,
        label_names=None,
        **kwargs
    ):
        super().__init__(**kwargs)

        # Model architecture parameters
        self.base_model = base_model
        self.use_hierarchy = use_hierarchy
        self.gated_hierarchy = gated_hierarchy
        self.gate_threshold = gate_threshold
        self.hidden_size = hidden_size
        self.hierarchical_hidden_size = hierarchical_hidden_size
        self.num_hidden_layers = num_hidden_layers
        self.dropout = dropout
        self.attention_dropout = attention_dropout

        # Training parameters
        self.use_weighted_loss = use_weighted_loss
        self.use_focal_loss = use_focal_loss
        self.focal_gamma = focal_gamma
        self.teacher_forcing_ratio = teacher_forcing_ratio

        # NEW: Control main label prediction
        self.predict_main_labels = predict_main_labels

        # Label dimensions
        self.num_main_labels = num_main_labels
        self.num_event_labels = num_event_labels
        self.num_cause_labels = num_cause_labels
        self.num_action_labels = num_action_labels
        self.total_labels = total_labels
        self.label_indices = label_indices or {}
        self.label_names = label_names or []


class ConfigurableHierarchicalModel(PreTrainedModel):
    """Highly configurable hierarchical multi-label classifier"""
    config_class = ConfigurableHierarchicalConfig
    base_model_prefix = "configurable_hierarchical"
    supports_gradient_checkpointing = True

    def __init__(self, config: ConfigurableHierarchicalConfig):
        super().__init__(config)

        # Store config
        self.config = config

        # Load base encoder
        self.encoder = AutoModel.from_pretrained(config.base_model)

        # Apply additional dropout to encoder if specified
        if hasattr(config, 'attention_dropout') and config.attention_dropout > 0:
            self.encoder.config.attention_probs_dropout_prob = config.attention_dropout

        # Main classifiers (ONLY if predict_main_labels is True)
        if config.predict_main_labels and config.num_main_labels > 0:
            self.main_classifier = nn.Linear(config.hidden_size, config.num_main_labels)
        else:
            self.main_classifier = None

        # Build sublabel classifiers based on configuration
        if config.use_hierarchy and config.predict_main_labels:
            # Hierarchical: sublabels depend on main labels
            hierarchical_input_size = config.hidden_size + config.num_main_labels
        else:
            # Non-hierarchical or no main labels: sublabels independent
            hierarchical_input_size = config.hidden_size

        # Create sublabel classifiers with configurable depth
        self.event_classifier = self._build_sublabel_classifier(
            hierarchical_input_size,
            config.num_event_labels,
            config.hierarchical_hidden_size,
            config.num_hidden_layers,
            config.dropout
        )

        self.cause_classifier = self._build_sublabel_classifier(
            hierarchical_input_size,
            config.num_cause_labels,
            config.hierarchical_hidden_size,
            config.num_hidden_layers,
            config.dropout
        )

        self.action_classifier = self._build_sublabel_classifier(
            hierarchical_input_size,
            config.num_action_labels,
            config.hierarchical_hidden_size,
            config.num_hidden_layers,
            config.dropout
        )

        # Store config for forward pass
        self.use_hierarchy = config.use_hierarchy
        self.gated_hierarchy = config.gated_hierarchy
        self.gate_threshold = config.gate_threshold
        self.predict_main_labels = config.predict_main_labels

        # Initialize weights
        self.post_init()

    def _build_sublabel_classifier(self, input_size, output_size, hidden_size, num_layers, dropout):
        """Build a sublabel classifier with configurable depth"""
        if output_size == 0:
            return None

        layers = []

        for i in range(num_layers):
            if i == 0:
                layers.append(nn.Linear(input_size, hidden_size))
            else:
                layers.append(nn.Linear(hidden_size, hidden_size))

            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dropout))

        # Output layer
        layers.append(nn.Linear(hidden_size, output_size))

        return nn.Sequential(*layers)

    def forward(
        self,
        input_ids=None,
        attention_mask=None,
        labels=None,
        teacher_forcing=False,
        return_dict=None,
        **kwargs
    ):
        return_dict = return_dict if return_dict is not None else self.config.use_return_dict

        # Get encoder outputs
        encoder_outputs = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask,
            return_dict=True
        )

        # Pool the outputs
        pooled_output = encoder_outputs.last_hidden_state[:, 0]

        # Get main predictions (only if enabled)
        if self.main_classifier is not None:
            main_logits = self.main_classifier(pooled_output)

            if self.use_hierarchy:
                # Use main predictions for sublabel input
                if teacher_forcing and labels is not None:
                    main_probs = labels[:, :self.config.num_main_labels].float()
                else:
                    main_probs = torch.sigmoid(main_logits)

                hierarchical_input = torch.cat([pooled_output, main_probs], dim=1)
            else:
                # Non-hierarchical: use only pooled output
                hierarchical_input = pooled_output
        else:
            # No main labels - use pooled output directly
            main_logits = torch.zeros(pooled_output.shape[0], 0).to(pooled_output.device)
            hierarchical_input = pooled_output

        # Get sublabel predictions
        event_logits = self.event_classifier(hierarchical_input) if self.event_classifier else torch.zeros(main_logits.shape[0], 0).to(main_logits.device)
        cause_logits = self.cause_classifier(hierarchical_input) if self.cause_classifier else torch.zeros(main_logits.shape[0], 0).to(main_logits.device)
        action_logits = self.action_classifier(hierarchical_input) if self.action_classifier else torch.zeros(main_logits.shape[0], 0).to(main_logits.device)

        # Apply gating if configured (only if we have main labels)
        if self.gated_hierarchy and self.use_hierarchy and self.main_classifier is not None:
            main_probs = torch.sigmoid(main_logits)

            # Gate EVENT sublabels
            if event_logits.shape[1] > 0:
                event_gate = torch.where(
                    main_probs[:, 0:1] > self.gate_threshold,
                    torch.ones_like(main_probs[:, 0:1]),
                    torch.zeros_like(main_probs[:, 0:1])
                )
                event_logits = event_logits * event_gate

            # Gate CAUSE sublabels
            if cause_logits.shape[1] > 0:
                cause_gate = torch.where(
                    main_probs[:, 1:2] > self.gate_threshold,
                    torch.ones_like(main_probs[:, 1:2]),
                    torch.zeros_like(main_probs[:, 1:2])
                )
                cause_logits = cause_logits * cause_gate

            # Gate ACTION sublabels
            if action_logits.shape[1] > 0:
                action_gate = torch.where(
                    main_probs[:, 2:3] > self.gate_threshold,
                    torch.ones_like(main_probs[:, 2:3]),
                    torch.zeros_like(main_probs[:, 2:3])
                )
                action_logits = action_logits * action_gate

        # Concatenate all logits (only include main_logits if they exist)
        if self.main_classifier is not None:
            logits = torch.cat([
                main_logits, event_logits, cause_logits, action_logits
            ], dim=1)
        else:
            logits = torch.cat([
                event_logits, cause_logits, action_logits
            ], dim=1)

        # Calculate loss if labels provided
        loss = None
        if labels is not None:
            if self.config.use_focal_loss and hasattr(self.config, 'focal_gamma'):
                loss = self._focal_loss(logits, labels.float(), gamma=self.config.focal_gamma)
            else:
                loss_fct = nn.BCEWithLogitsLoss()
                loss = loss_fct(logits, labels.float())

        if not return_dict:
            output = (logits,) + encoder_outputs[2:]
            return ((loss,) + output) if loss is not None else output

        return SequenceClassifierOutput(
            loss=loss,
            logits=logits,
            hidden_states=encoder_outputs.hidden_states,
            attentions=encoder_outputs.attentions,
        )

    def _focal_loss(self, logits, targets, gamma=2.0):
        """Focal loss for handling extreme class imbalance"""
        bce_loss = nn.functional.binary_cross_entropy_with_logits(
            logits, targets, reduction='none'
        )
        probas = torch.sigmoid(logits)

        # Calculate focal weights
        focal_weight = torch.where(
            targets == 1,
            (1 - probas) ** gamma,
            probas ** gamma
        )

        focal_loss = focal_weight * bce_loss
        return focal_loss.mean()

In [15]:
# ============================================================================
# CELL 18: TEST LOADING THE MODEL
# ============================================================================

# Clear the model from memory
# del model

output_dir = "models/training_20251014_124910"



# Save the model using Hugging Face's save_pretrained
final_model_path = f"{output_dir}/final_model"

# Load the model using Hugging Face's Auto classes
from transformers import AutoTokenizer

# First register the classes if loading in a new session
AutoConfig.register("configurable_hierarchical", ConfigurableHierarchicalConfig)
AutoModel.register(ConfigurableHierarchicalConfig, ConfigurableHierarchicalModel)

# Load model and tokenizer
loaded_model = AutoModel.from_pretrained(final_model_path)
loaded_tokenizer = AutoTokenizer.from_pretrained(final_model_path)

print("✅ Model successfully loaded using AutoModel!")

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of the model checkpoint at models/training_20251014_124910/final_model were not used when initializing ConfigurableHierarchicalModel: ['main_classifier.bias', 'main_classifier.weight']
- This IS expected if you are initializing ConfigurableHierarchicalModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing ConfigurableHierarchicalModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


✅ Model successfully loaded using AutoModel!


In [16]:
def Hierarchy_predictor(data, model, tokenizer, label_names, thresholds=None):
    """Quick function to predict labels for a new passage"""
    dataOutput = []
    for text in data:
        # Tokenize
        inputs = tokenizer(
            text['passage'],
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=512
        ).to(model.device)

        #Get actual labels
        actual_labels = [text[label] for label in label_names]

        # Predict
        with torch.no_grad():
            outputs = model(**inputs)
            probs = torch.sigmoid(outputs.logits).cpu().numpy()[0]

        # Get probabilities
        prob_dict = {
            label: float(probs[i])
            for i, label in enumerate(label_names)
        }

        # Apply thresholds
        if thresholds:
            pred_labels = [1 if float(probs[i]) > thresholds.get(label, {}).get('threshold', 0.5) else 0 
                        for i, label in enumerate(label_names)]  
        else:
            pred_labels = [1 if float(probs[i]) > 0.5 else 0 
                for i, label in enumerate(label_names)]



        output_dict = dict()
        output_dict["pred_labels"] = pred_labels
        output_dict["actual_labels"] = actual_labels
        output_dict["passage"] = text['passage']
        output_dict["ID"] = text['ID']
        dataOutput.append(output_dict)
    return dataOutput




In [17]:
# EC Load optimal thresholds and labels
f = open(final_model_path+"/training_info.json")
data_loaded = json.load(f)
f.close()

optimal_thresholds = data_loaded["optimal_thresholds"]
# # get label names (if using John's dataset, uncomment the enxt line of code and get it directly from the training set, otherwise if using the HRAF style column types, use the next code)
label_columns = data_loaded["label_columns"]
assert label_columns == [label for label in Hraf.features.keys() if label not in ['ID', 'passage']], "labels do not match or are in the wrong order"
# label_columns = [label for label in Hraf.features.keys() if label not in ['ID', 'passage']]
label_columns

['Illness',
 'Accident',
 'Other',
 'Material_Physical',
 'Spirits_Gods',
 'Witchcraft_Sorcery',
 'Rule_Violation_Taboo',
 'Physical_Material',
 'Technical_Specialist',
 'Divination',
 'Shaman_Medium_Healer',
 'Priest_High_Religion']

In [18]:
# run the inference
HrafOutput = Hierarchy_predictor(
    Hraf,
    loaded_model,  # Changed from 'model' to 'loaded_model'
    loaded_tokenizer,  # Changed from 'tokenizer' to 'loaded_tokenizer' for consistency
    label_columns,
    optimal_thresholds
)

In [19]:
df_score = score(HrafOutput, label_columns) 
df_score

,Illness_F1,Accident_F1,Other_F1,Material_Physical_F1,Spirits_Gods_F1,Witchcraft_Sorcery_F1,Rule_Violation_Taboo_F1,Physical_Material_F1,Technical_Specialist_F1,Divination_F1,Shaman_Medium_Healer_F1,Priest_High_Religion_F1,Micro_F1,Macro_F1,Weighted_F1
NLP,0.65,0.633,0.547,0.589,0.637,0.751,0.68,0.498,0.627,0.61,0.449,0.374,0.582,0.587,0.591


## CHi Square

In [41]:
from scipy.stats import chi2_contingency

ct_EVENT_CAUSE = pd.crosstab(df[('EVENT','No_Info')], df[('CAUSE','No_Info')], rownames=['ACTION'], colnames=['CAUSE'])
ct_EVENT_CAUSE

array([[1167,  351],
       [  49,  183]], dtype=int64)

In [119]:
def chi_square_calc(row, col):
    cross_tab = pd.crosstab(df[(row,'No_Info')], df[(col,'No_Info')], rownames=[row], colnames=[col])
    stat, p, dof, expected = chi2_contingency(cross_tab)
    results = f"{row} by {col}:\nchi: {round(stat,1)}\np:   {round(p,3)}\n\n"
    return results

group_list = [('EVENT', 'CAUSE'), ('EVENT', 'ACTION'), ('ACTION', 'CAUSE')]
for row, col in group_list:
    print(chi_square_calc(row, col))

EVENT by CAUSE:
chi: 292.4
p:   0.0


EVENT by ACTION:
chi: 103.3
p:   0.0


ACTION by CAUSE:
chi: 0.0
p:   0.857




In [44]:
def chi_sqr(obs):
    size_x = obs.shape
    chi_mat = np.zeros(size_x)
    for row in range(size_x[0]):
        for col in range(size_x[1]):
            exp = np.sum(x[row]) * np.sum(x[:,col]) / np.sum(x)
            chi_mat[row, col] = np.sum((obs[row, col] - exp)**2 / exp)
    return chi_mat

print(np.sum(chi_sqr(x)))
